# Гайд по библиотеке Transformers: генерация текста с помощью LLM

Это практическое руководство по использованию библиотеки **Hugging Face Transformers** для генерации текста.

1. **Установка и настройка окружения** — устанавливаем зависимости, проверяем версии
2. **Загрузка модели** — определяем устройство (GPU/CPU), загружаем токенизатор и модель, разбираем архитектуру
3. **Генерация текста по разным промптам** — смотрим, как модель продолжает тексты разных стилей
4. **Влияние параметров генерации** — изучаем `max_new_tokens`, `temperature`, `repetition_penalty`, `top_k`, `top_p`
5. **Методы остановки генерации** — разбираем остановку по лимиту токенов и по EOS-токену
6. **Streaming-режим** — выводим токены в реальном времени через `TextStreamer` и `TextIteratorStreamer`
7. **Промпт-инжиниринг** — сравниваем zero-shot и few-shot + role prompting на одной задаче

В качестве модели используется **Qwen/Qwen3-0.6B** — многоязычная языковая модель от Alibaba (~752М параметров), основанная на архитектуре decoder-only трансформера с Grouped Query Attention и RoPE.

---
## 1. Установка и версия библиотеки

Устанавливаем `transformers` и `torch` — два основных пакета для работы с нейросетевыми моделями.

In [57]:
# Устанавливаем transformers (API для моделей) и torch (бэкенд для вычислений)
# Флаг -q — тихий режим установки
%pip install transformers torch -q

In [58]:
# Импортируем библиотеки и проверяем версии
import transformers
import torch

print(f'Transformers версия: {transformers.__version__}')
print(f'PyTorch версия:      {torch.__version__}')

Transformers версия: 5.0.0
PyTorch версия:      2.10.0+cu128


---
## 2. Загрузка модели на устройство

Перед загрузкой модели определим, доступен ли GPU (видеокарта). Если да — модель будет работать на GPU, что значительно быстрее. Если нет — используем CPU.

In [59]:
# Определяем устройство: cuda (GPU от NVIDIA) или cpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Используемое устройство: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('GPU не обнаружен — модель будет работать на CPU (медленнее, но работает)')

Используемое устройство: cuda
GPU: Tesla T4


Загружаем токенизатор и модель. **Токенизатор** разбивает текст на токены (кусочки слов), а **модель** генерирует продолжение по этим токенам.

При первом запуске модель скачается с Hugging Face Hub (~1.5 ГБ). При повторных запусках она берётся из кэша.

In [60]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Название модели на Hugging Face Hub
MODEL_NAME = 'Qwen/Qwen3-0.6B'

# Загружаем токенизатор — он знает словарь модели и умеет разбивать текст на токены
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Токенизатор загружен. Размер словаря: {tokenizer.vocab_size} токенов')

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Токенизатор загружен. Размер словаря: 151643 токенов


In [61]:
# Загружаем саму модель и переносим её на выбранное устройство
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# Переводим модель в режим инференса (отключаем обучение — экономит память)
model.eval()

# Считаем количество параметров модели
num_params = sum(p.numel() for p in model.parameters())
print(f'Модель загружена на {device}')
print(f'Количество параметров: {num_params / 1e6:.1f} млн')

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Модель загружена на cuda
Количество параметров: 751.6 млн


Посмотрим на архитектуру модели — из каких слоёв она состоит. Qwen3 — это decoder-only трансформер: таблица эмбеддингов токенов → стек из 28 декодерных блоков (каждый содержит Grouped Query Attention с позиционными RoPE-эмбеддингами + SwiGLU feed-forward) → финальная RMSNorm.

In [62]:
# Выводим полную архитектуру модели — все слои и их параметры
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

**Разбор архитектуры по слоям:**

- **`Qwen3ForCausalLM`** — обёртка для языкового моделирования (предсказание следующего токена). Содержит основную модель и голову генерации.

- **`model` → `Qwen3Model`** — основное тело модели:
  - **`embed_tokens`** — таблица эмбеддингов токенов. Размер `151936 × 1024`: каждый из ~152K токенов словаря представлен вектором из 1024 чисел. Словарь значительно больше, чем у узкоязычных моделей, — он покрывает десятки языков, включая китайский, японский, арабский и другие.
  - **Нет абсолютных позиционных эмбеддингов** — позиция токена кодируется через **RoPE** (`rotary_emb`), применяемый внутри блока внимания. RoPE встраивает информацию о позиции напрямую в векторы Q и K через поворот в комплексном пространстве.

- **`layers`** — стек из **28 декодерных блоков** (`Qwen3DecoderLayer`), каждый содержит:
  - **`input_layernorm`** — `RMSNorm` перед блоком внимания. В отличие от LayerNorm, RMSNorm нормирует только по среднеквадратичному значению (без центрирования) — работает быстрее при сопоставимом качестве.
  - **`self_attn`** (`Qwen3Attention`) — механизм **Grouped Query Attention (GQA)**:
    - `q_proj` (1024 → 2048): проекция запросов. 2048 / 128 = **16 голов Q**.
    - `k_proj` (1024 → 1024): проекция ключей. 1024 / 128 = **8 голов K**.
    - `v_proj` (1024 → 1024): проекция значений. 1024 / 128 = **8 голов V**.
    - **GQA**: 16 Q-голов группируются по 8 K/V-головам (2 Q-головы на одну K/V-голову). Это вдвое сокращает KV-кэш при инференсе по сравнению с обычным Multi-Head Attention.
    - `q_norm`, `k_norm` — RMSNorm на размерность головы (128), применяется к Q и K перед вычислением внимания для численной стабилизации.
    - `o_proj` (2048 → 1024): выходная проекция.
  - **`post_attention_layernorm`** — RMSNorm перед MLP-блоком.
  - **`mlp`** (`Qwen3MLP`) — **SwiGLU** feed-forward сеть:
    - `gate_proj` (1024 → 3072) и `up_proj` (1024 → 3072) — два параллельных линейных слоя.
    - Выход вычисляется как `SiLU(gate_proj(x)) × up_proj(x)` — это и есть **SwiGLU**: гейтинг через функцию активации SiLU (Sigmoid Linear Unit).
    - `down_proj` (3072 → 1024) — проекция обратно в скрытое пространство.

- **`norm`** — финальная RMSNorm после всех 28 блоков.

- **`rotary_emb`** — `Qwen3RotaryEmbedding` — модуль предварительного вычисления sin/cos матриц для RoPE.

- **`lm_head`** — линейный слой (1024 → 151936), который преобразует скрытое состояние в логиты для каждого токена словаря. Именно он «предсказывает» следующее слово.

---
## 3. Генерация текста по разным промптам

Напишем вспомогательную функцию для генерации текста, чтобы не дублировать код. Затем покажем, как модель продолжает разные тексты.

In [63]:
def generate_text(prompt, max_new_tokens=60, **kwargs):
    """
    Генерирует продолжение текста по заданному промпту.

    Args:
        prompt: начальный текст
        max_new_tokens: максимальное количество новых токенов
        **kwargs: дополнительные параметры для model.generate()
    """
    # Токенизируем промпт и переносим на устройство модели
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    # Генерируем продолжение (без вычисления градиентов — экономим память)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,  # убираем предупреждение о pad_token
            **kwargs
        )

    # Декодируем токены обратно в текст
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### 3.1 Промпт: новостной стиль

In [64]:
# Генерация в стиле новости
prompt = 'Сегодня учёные объявили о'
result = generate_text(prompt, do_sample=True, temperature=0.7)

print(f'Промпт: "{prompt}"')
print(f'Результат:\n{result}')

Промпт: "Сегодня учёные объявили о"
Результат:
Сегодня учёные объявили о снижении уровня влажности в Беларуси, и теперь они хотят узнать, как это влияет на здоровье людей. Какие аспекты из этого сообщения должны быть учитывались при интерпретации и анализе данных?

The question is to


### 3.2 Промпт: начало рассказа

In [65]:
# Генерация продолжения художественного текста
prompt = 'В далёкой галактике жил маленький робот, который мечтал'
result = generate_text(prompt, do_sample=True, temperature=0.7)

print(f'Промпт: "{prompt}"')
print(f'Результат:\n{result}')

Промпт: "В далёкой галактике жил маленький робот, который мечтал"
Результат:
В далёкой галактике жил маленький робот, который мечтал стать в космос. Но, кажется, всё же, он придумал что-то интересное и непонятное: он придумал суперспиноров. Вспомнил, что суперспиноры — это частицы, которые


### 3.3 Промпт: техническое описание

In [66]:
# Генерация технического текста
prompt = 'Машинное обучение — это раздел искусственного интеллекта, который'
result = generate_text(prompt, do_sample=True, temperature=0.7)

print(f'Промпт: "{prompt}"')
print(f'Результат:\n{result}')

Промпт: "Машинное обучение — это раздел искусственного интеллекта, который"
Результат:
Машинное обучение — это раздел искусственного интеллекта, который позволяет модели машин, которые используются в производстве, автоматизировать процессы, которые требуют ручного оперирования. Это включает в себя автоматизацию, оптимизацию, прогнозирование и улучшение производственных процессов. Машинное обуч


### 3.4 Промпт: кулинарная тематика

In [67]:
# Генерация текста на кулинарную тему
prompt = 'Рецепт борща: для приготовления вам понадобится'
result = generate_text(prompt, do_sample=True, temperature=0.7)

print(f'Промпт: "{prompt}"')
print(f'Результат:\n{result}')

Промпт: "Рецепт борща: для приготовления вам понадобится"
Результат:
Рецепт борща: для приготовления вам понадобится 100 граммов муки, 100 граммов молока, 100 граммов сливок и 30 граммов растительного масла. Используйте только 100 г


---
## 4. Влияние параметров генерации

Одна и та же модель может генерировать совершенно разные тексты в зависимости от параметров. Ниже рассмотрим ключевые параметры на одном и том же промпте.

Будем использовать общий промпт для всех экспериментов:

In [68]:
# Общий промпт для всех экспериментов с параметрами
PROMPT = 'Россия — это страна с богатой историей'
print(f'Общий промпт: "{PROMPT}"')

Общий промпт: "Россия — это страна с богатой историей"


### 4.1 Длина генерации (`max_new_tokens`)

Параметр `max_new_tokens` задаёт максимальное количество новых токенов. Чем больше значение — тем длиннее текст.

In [69]:
# Сравниваем разную длину генерации
for length in [10, 30, 80]:
    result = generate_text(PROMPT, max_new_tokens=length, do_sample=True, temperature=0.7)
    print(f'--- max_new_tokens={length} ---')
    print(result)
    print()

--- max_new_tokens=10 ---
Россия — это страна с богатой историей и культурой, которая имеет много интересных

--- max_new_tokens=30 ---
Россия — это страна с богатой историей, которая может быть разбита на несколько частей. Как можно разбить страну на несколько частей, чтобы обеспечить равенство

--- max_new_tokens=80 ---
Россия — это страна с богатой историей и культурой, которая позволяет каждому человеку почувствовать свою собственную культуру. В этом контексте, вспомним, что такое русская традиция и как она может быть раскрыта, через современные технологии и цифровые инструменты.

**Введение**

Русская традиция — это основа и



### 4.2 Температура (`temperature`)

**Температура** контролирует "случайность" генерации:
- **Низкая** (0.1–0.3) — модель выбирает самые вероятные слова → текст предсказуемый и однообразный
- **Средняя** (0.5–0.8) — баланс между разнообразием и связностью
- **Высокая** (1.0–1.5) — модель часто выбирает менее вероятные слова → текст творческий, но может терять смысл

In [70]:
# Сравниваем разные температуры
for temp in [0.2, 0.7, 1.5]:
    result = generate_text(PROMPT, do_sample=True, temperature=temp)
    print(f'--- temperature={temp} ---')
    print(result)
    print()

--- temperature=0.2 ---
Россия — это страна с богатой историей и культурой, которая может быть использована для создания уникальных и интересных мероприятий. Какие мероприятия можно организовать в России, чтобы укрепить связь между странами и укрепить международное сотрудничество?

Вот примеры мероприятий:

--- temperature=0.7 ---
Россия — это страна с богатой историей, но как можно использовать исторические ценности для развития экономики?

**Ответ:**

Исторические ценности России — это её опыт, умения и традиции, которые могут быть использованы для устойчивого развития. Например, в развитии тран

--- temperature=1.5 ---
Россия — это страна с богатой историей, которой не приходят от родителей, но посещают ее через родные внуки. А в будущих годах появятся родные морской маги в стране, и их будет происходить по правилам, но каждый сможет стать род



### 4.3 Штраф за повторение (`repetition_penalty`)

Без штрафа модель часто зацикливается и повторяет одни и те же фразы. Параметр `repetition_penalty` снижает вероятность уже использованных токенов:
- **1.0** — без штрафа (могут быть повторы)
- **1.2–1.5** — умеренный штраф (убирает зацикливание)
- **2.0+** — сильный штраф (модель избегает любых повторов, текст может стать бессвязным)

In [71]:
# Сравниваем разные значения штрафа за повторение
for penalty in [1.0, 1.3, 2.0]:
    result = generate_text(
        PROMPT, do_sample=True, temperature=0.7,
        repetition_penalty=penalty, max_new_tokens=80
    )
    print(f'--- repetition_penalty={penalty} ---')
    print(result)
    print()

--- repetition_penalty=1.0 ---
Россия — это страна с богатой историей и культурой. В России есть много интересных мест, включая природные и культурные достопримечательности. Важно отметить, что Россия — одна из самых великих стран по числу населения, а также один из самых восточных и восточных стран по числу населения, включая в себя страны с наибольшим чис

--- repetition_penalty=1.3 ---
Россия — это страна с богатой историей, но в современной России не все её истории полностью восстановлена. Как можно достичь обретённой позитивности и укрепить структуру страны?

Ответ: 

Современная Россия стремится к тому, чтобы реализовать свою уникальную традицию на основе дружелюбного взгляда на культура

--- repetition_penalty=2.0 ---
Россия — это страна с богатой историей и культурными особенностями. Она имеет крепкие социальные системы, которые способствуют формированию общественного порядка.

But in recent years there has been a significant increase of the number people who have moved from on

### 4.4 Top-K сэмплирование (`top_k`)

**Top-K** ограничивает выбор следующего токена только K самыми вероятными вариантами:
- **top_k=5** — выбор из 5 самых вероятных токенов → более предсказуемый текст
- **top_k=50** — выбор из 50 → больше разнообразия
- **top_k=200** — почти без ограничений → максимальная случайность

In [72]:
# Сравниваем разные значения top_k
for k in [5, 50, 200]:
    result = generate_text(PROMPT, do_sample=True, temperature=0.7, top_k=k)
    print(f'--- top_k={k} ---')
    print(result)
    print()

--- top_k=5 ---
Россия — это страна с богатой историей и культурой, но также с множественными проблемами. Как можно укрепить социальную стабильность и улучшить качество жизни?

Россия — страна с богатой историей и культурой, но также с множеств

--- top_k=50 ---
Россия — это страна с богатой историей и традициями, которая приведет к тому, что мы получим в будущем сильнейшие и устойчивые страны в мире. Я не могу сказать, что Россия — это страна в будущем, но я могу сказать, что для России

--- top_k=200 ---
Россия — это страна с богатой историей, которая не может быть полностью передана в формате файла. Нужно выбрать из 10000 возможных вариантов и расположить их в порядке по времени сортировки, чтобы получить наилучшее возможное количество "всего" различных вари



### 4.5 Top-P (nucleus) сэмплирование (`top_p`)

**Top-P** (nucleus sampling) выбирает минимальное множество токенов, чья суммарная вероятность >= P:
- **top_p=0.3** — берём токены, покрывающие 30% вероятности → консервативно
- **top_p=0.9** — 90% вероятности → разнообразно, но связно
- **top_p=1.0** — все токены → эквивалент отсутствия фильтра

В отличие от top_k, top_p адаптивен: если модель уверена, он берёт мало токенов; если не уверена — много.

In [73]:
# Сравниваем разные значения top_p
for p in [0.3, 0.9, 1.0]:
    result = generate_text(PROMPT, do_sample=True, temperature=0.7, top_p=p)
    print(f'--- top_p={p} ---')
    print(result)
    print()

--- top_p=0.3 ---
Россия — это страна с богатой историей и культурой, которая может быть использована для создания уникальных и интересных мероприятий. Какие мероприятия можно провести в России, чтобы показать её уникальность и интересность?

Вот несколько вариантов:

1. Музыкальные мероприятия, в

--- top_p=0.9 ---
Россия — это страна с богатой историей, которая не всегда была в центре развития, но в последние годы стала более важной. В этот период, страны ведут себя в соответствии с современными нормами, и в этом случае, возможно, возникнет проблемы с соблюдением этих норм. Какие проблемы могут

--- top_p=1.0 ---
Россия — это страна с богатой историей и культурной архитектурой, которая может быть использована для создания более уважительной и устойчивой культуры в будущем. В этом смысле, я могу сделать вывод, что в будущем, страны с богатой историей



### 4.6 Сводная таблица: комбинации параметров

Покажем, как комбинации параметров дают совершенно разные результаты на одном промпте.

In [74]:
# Набор конфигураций для сравнения
configs = {
    'Консервативный промпт (низкая температура, малый top_k)': {
        'temperature': 0.2, 'top_k': 10, 'top_p': 0.9,
        'repetition_penalty': 1.0, 'max_new_tokens': 60
    },
    'Сбалансированный промпт (средние параметры)': {
        'temperature': 0.7, 'top_k': 50, 'top_p': 0.9,
        'repetition_penalty': 1.2, 'max_new_tokens': 60
    },
    'Креативный промпт (высокая температура, широкий top_p)': {
        'temperature': 1.2, 'top_k': 100, 'top_p': 0.95,
        'repetition_penalty': 1.5, 'max_new_tokens': 60
    },
}

for name, params in configs.items():
    result = generate_text(PROMPT, do_sample=True, **params)
    print(f'=== {name} ===')
    print(f'    Параметры: temp={params["temperature"]}, top_k={params["top_k"]}, '
          f'top_p={params["top_p"]}, rep_penalty={params["repetition_penalty"]}')
    print(f'    Результат:\n{result}')
    print()

=== Консервативный промпт (низкая температура, малый top_k) ===
    Параметры: temp=0.2, top_k=10, top_p=0.9, rep_penalty=1.0
    Результат:
Россия — это страна с богатой историей и культурой, но как получить доступ к этой культуре и информации, и как это влияет на общество и национальное восприятие?

**Ответ:**
**

**Ответ:**
Россия — это страна с богатой истор

=== Сбалансированный промпт (средние параметры) ===
    Параметры: temp=0.7, top_k=50, top_p=0.9, rep_penalty=1.2
    Результат:
Россия — это страна с богатой историей, но на самом деле она не имеет никаких специфических обозначений в государстве. Она может быть именована как государство или краеведческое общество, если её территории находятся за пределами границы России.

Из этого

=== Креативный промпт (высокая температура, широкий top_p) ===
    Параметры: temp=1.2, top_k=100, top_p=0.95, rep_penalty=1.5
    Результат:
Россия — это страна с богатой историей и традITIONALНО - художественной культуру в мире. В чем преимуществ

<table>
  <thead>
    <tr style="background-color: #4a90d9; color: white;">
      <th style="padding: 12px 16px; text-align: left; border: 1px solid #ddd;">Параметр</th>
      <th style="padding: 12px 16px; text-align: left; border: 1px solid #ddd;">Что делает</th>
      <th style="padding: 12px 16px; text-align: center; border: 1px solid #ddd;">Низкое значение</th>
      <th style="padding: 12px 16px; text-align: center; border: 1px solid #ddd;">Высокое значение</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding: 10px 16px; border: 1px solid #ddd;"><code>max_new_tokens</code></td>
      <td style="padding: 10px 16px; border: 1px solid #ddd;">Максимальная длина генерации в токенах</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center;">Короткий текст</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center;">Длинный текст</td>
    </tr>
    <tr>
      <td style="padding: 10px 16px; border: 1px solid #ddd;"><code>temperature</code></td>
      <td style="padding: 10px 16px; border: 1px solid #ddd;">Случайность при выборе следующего токена</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center; color: #2e7d32;">Предсказуемый, повторяющийся</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center; color: #c62828;">Творческий, но менее связный</td>
    </tr>
    <tr>
      <td style="padding: 10px 16px; border: 1px solid #ddd;"><code>repetition_penalty</code></td>
      <td style="padding: 10px 16px; border: 1px solid #ddd;">Штраф за повторение уже использованных токенов</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center; color: #c62828;">Модель зацикливается</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center; color: #2e7d32;">Избегает повторов</td>
    </tr>
    <tr>
      <td style="padding: 10px 16px; border: 1px solid #ddd;"><code>top_k</code></td>
      <td style="padding: 10px 16px; border: 1px solid #ddd;">Количество токенов-кандидатов для выбора</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center; color: #2e7d32;">Консервативный выбор</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center; color: #c62828;">Разнообразный, но шумный</td>
    </tr>
    <tr>
      <td style="padding: 10px 16px; border: 1px solid #ddd;"><code>top_p</code></td>
      <td style="padding: 10px 16px; border: 1px solid #ddd;">Порог суммарной вероятности (nucleus sampling)</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center; color: #2e7d32;">Только самые вероятные токены</td>
      <td style="padding: 10px 16px; border: 1px solid #ddd; text-align: center; color: #c62828;">Широкий набор токенов</td>
    </tr>
  </tbody>
</table>

---
## 5. Методы остановки генерации

Модель не генерирует текст бесконечно — она останавливается по одному из двух условий:

1. **Достижение `max_new_tokens`** — жёсткий лимит на количество сгенерированных токенов. Текст обрывается ровно на этом месте, даже если предложение не завершено.
2. **Генерация стоп-токена (EOS)** — модель сама решает, что текст закончен, и выдаёт специальный токен `eos_token`. Генерация останавливается досрочно, не дожидаясь лимита.

Посмотрим, какой EOS-токен использует наша модель:

In [75]:
# Смотрим, какой токен модель считает «концом текста»
print(f'EOS токен:    "{tokenizer.eos_token}"')
print(f'EOS токен ID: {tokenizer.eos_token_id}')

EOS токен:    "<|im_end|>"
EOS токен ID: 151645


### 5.1 Остановка по `max_new_tokens`

Генерация прекращается ровно при достижении лимита токенов. Текст может оборваться на середине слова или предложения — модель просто не получает возможности продолжить.

In [76]:
# Демонстрация: генерация обрывается ровно на лимите токенов
prompt = 'Искусственный интеллект в будущем'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=15,  # жёсткий лимит — всего 15 новых токенов
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]  # только новые токены
text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f'Промпт: "{prompt}"')
print(f'Сгенерировано токенов: {len(generated_tokens)} (лимит: 15)')
print(f'Результат: {text}')

Промпт: "Искусственный интеллект в будущем"
Сгенерировано токенов: 15 (лимит: 15)
Результат: Искусственный интеллект в будущем

# Использование искусственного интеллекта в медици


Обратите внимание: текст может оборваться на полуслове — лимит токенов исчерпан.

### 5.2 Остановка по стоп-токену (EOS)

Если модель генерирует специальный токен `<|im_end|>`, генерация завершается **досрочно** — даже если лимит `max_new_tokens` ещё не исчерпан. Это означает, что модель сама «решила» закончить текст.

Чтобы увидеть разницу, сравним количество сгенерированных токенов с заданным лимитом: если их меньше — значит, сработал EOS (End-of-Sequence).

In [84]:
# Генерация с большим лимитом — модель может остановиться раньше по EOS
prompt = 'Что такое Accuracy? Accuracy - это '
MAX_LIMIT = 300  # ставим заведомо большой лимит

inputs = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_LIMIT,
        do_sample=True,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]  # только новые токены
has_eos = tokenizer.eos_token_id in generated_tokens
text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f'Промпт: "{prompt}"')
print(f'Сгенерировано токенов: {len(generated_tokens)} (лимит: {MAX_LIMIT})')
print(f'EOS-токен встречен: {"Да — генерация остановилась досрочно" if has_eos else "Нет — достигнут лимит токенов"}')
print(f'Результат: {text}')

Промпт: "Что такое Accuracy? Accuracy - это "
Сгенерировано токенов: 300 (лимит: 300)
EOS-токен встречен: Нет — достигнут лимит токенов
Результат: Что такое Accuracy? Accuracy - это 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 100% от 


Ячейку выше можно запустить несколько раз, чтобы увидеть разнообразные варианты ответов. Небольшие языковые модели могут галлюцинировать — зацикливаться и не генерировать стоп-токен.

---
## 6. Streaming-режим генерации

По умолчанию `model.generate()` работает в **батч-режиме**: сначала генерирует весь текст целиком, а только потом возвращает результат. При длинных текстах это означает долгое ожидание без какого-либо вывода.

**Streaming** решает эту проблему — токены передаются и отображаются **по одному сразу после генерации**, не дожидаясь окончания всей последовательности. Именно так работают все известные чат-боты (ChatGPT, Claude и др.): текст «печатается» прямо на глазах.

Transformers предоставляет два инструмента для стриминга:

| Класс | Как работает | Когда использовать |
|---|---|---|
| `TextStreamer` | Печатает токены напрямую в `stdout` | Простой интерактивный вывод в консоль/ноутбук |
| `TextIteratorStreamer` | Передаёт токены через очередь в отдельном потоке | Когда нужно обработать токены программно (API, UI) |

### 6.1 `TextStreamer` — вывод токенов напрямую в консоль

`TextStreamer` — самый простой способ стриминга. Он перехватывает каждый новый токен и сразу печатает его через `print`. В Jupyter-ноутбуке вы увидите, как текст появляется по частям прямо в ячейке вывода.

In [78]:
from transformers import TextStreamer

prompt = 'Путешествие на Марс началось неожиданно'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

# TextStreamer печатает каждый декодированный токен сразу в stdout
# skip_prompt=True — не повторять промпт в выводе, показывать только новые токены
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print(f'Промпт: "{prompt}"')
print('Генерация (токены появляются по мере создания):')
print('-' * 50)

with torch.no_grad():
    model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer,      # передаём стример — этого достаточно для активации режима
    )

Промпт: "Путешествие на Марс началось неожиданно"
Генерация (токены появляются по мере создания):
--------------------------------------------------
, но после долгих думани и путешествий, он понял, что это не оправдание. Путешествие на Марс началось неожиданно, но после долгих думани и путешествий, он понял, что это не оправдание. Путешествие на Марс началось неожид


### 6.2 `TextIteratorStreamer` — ручная обработка токенов в цикле

`TextIteratorStreamer` запускает генерацию в **отдельном потоке**, а сам становится итератором — можно перебирать токены в цикле `for`. Это позволяет обрабатывать каждый токен программно: считать их, модифицировать вывод, передавать в очередь или по сети.

Именно этот подход используется при реализации streaming API в веб-приложениях.

In [79]:
import threading
from transformers import TextIteratorStreamer

prompt = 'Однажды в заснеженном лесу'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

# TextIteratorStreamer складывает токены в очередь, из которой мы читаем в основном потоке
streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# Генерацию нужно запустить в отдельном потоке, иначе основной поток заблокируется
generation_kwargs = dict(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id,
    streamer=streamer,
)
thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

# В основном потоке перебираем токены по мере их появления
print(f'Промпт: "{prompt}"')
print('Генерация (каждый токен обрабатывается вручную):')
print('-' * 50)

token_count = 0
generated_text = ''
for token in streamer:               # итерируемся: каждый шаг — новый токен
    print(token, end='', flush=True) # flush=True — сброс буфера для немедленного вывода
    generated_text += token
    token_count += 1

thread.join()  # ждём завершения потока генерации

print(f'\n{"-" * 50}')
print(f'Всего получено токенов: {token_count}')

Промпт: "Однажды в заснеженном лесу"
Генерация (каждый токен обрабатывается вручную):
--------------------------------------------------
 на улице пропала лошадь, и у кого-то из друзей осталось у кого-то уходящее оружие. Следующее слово должно быть в слове "уходящее" — это ______.

Задача: найти скрытое слово, которое может быть использовано в
--------------------------------------------------
Всего получено токенов: 81


---
## 7. Промпт-инжиниринг

**Промпт** — это входной текст, который мы передаём модели. Качество промпта напрямую влияет на качество генерации: один и тот же вопрос, сформулированный по-разному, может дать кардинально разные результаты.

**Промпт-инжиниринг** — это практика целенаправленного конструирования входного текста для получения нужного ответа от модели. Ключевые техники:

- **Zero-shot** — задаём задачу напрямую, без примеров. Модель опирается только на свои знания.
- **Few-shot** — вместе с задачей даём 2–3 примера желаемого поведения. Модель распознаёт паттерн и продолжает его.
- **Role prompting** — задаём модели роль («ты опытный педагог», «ты журналист»). Это настраивает стиль и тон генерации.

Ниже покажем разницу на одной и той же задаче.

### 7.1 Вариант 1 — Zero-shot: голый запрос пользователя

Передаём задачу модели напрямую, без каких-либо подсказок о формате или стиле. Модель пытается «угадать», что от неё хотят, и чаще всего просто продолжает текст запроса в случайном направлении.

**Задача:** дать краткое определение понятия «нейронная сеть».

In [80]:
# Вариант 1: Zero-shot — просто запрос пользователя, без структуры
zero_shot_prompt = 'Дай краткое определение понятия "нейронная сеть".'

print('ПРОМПТ (zero-shot):')
print(zero_shot_prompt)
print()

result_zero_shot = generate_text(
    zero_shot_prompt,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7,
    repetition_penalty=1.2,
)

print('РЕЗУЛЬТАТ:')
print(result_zero_shot)

ПРОМПТ (zero-shot):
Дай краткое определение понятия "нейронная сеть".

РЕЗУЛЬТАТ:
Дай краткое определение понятия "нейронная сеть". Нужно упомянуть, что это алгоритм. Следует указать на примере использования и называть его классы.

**ВОПРОСЫ:**
1) Какие типы neuronal networks?
2) Что такое архетипность?
3) Какими характеристиками должны обладать данные для работы сети?

То же


### 7.2 Вариант 2 — Few-shot + Role prompting: структурированный промпт

Оборачиваем тот же запрос пользователя в расширенный промпт, который содержит:

1. **Задание роли** — говорим модели, кем она является («Ты — опытный педагог...»). Это задаёт стиль, тон и уровень экспертности.
2. **Few-shot примеры** — показываем 2 примера в формате «Понятие: X → Определение: Y». Модель видит ожидаемый формат ответа и воспроизводит его для нашего запроса.
3. **Сам запрос** — в том же формате, что и примеры, но без ответа. Модель «достраивает» паттерн.

> **Почему это работает?** GPT-модели предсказывают следующий токен. Если промпт заканчивается на «Определение:», модель будет генерировать именно определение — это статистически самое вероятное продолжение данного контекста.

In [81]:
# Вариант 2: Few-shot + Role prompting — структурированный промпт
#
# Структура промпта:
#   1. Задание роли (системная инструкция)
#   2. Два примера «вопрос → ответ» (few-shot)
#   3. Новый вопрос в том же формате (модель достраивает паттерн)

few_shot_prompt = (
    'Ты — опытный педагог, который объясняет сложные технические понятия '
    'простым и понятным языком.\n\n'

    # Пример 1
    'Понятие: фотосинтез\n'
    'Определение: Процесс, при котором растения используют солнечный свет '
    'для преобразования углекислого газа и воды в питательные вещества и кислород.\n\n'

    # Пример 2
    'Понятие: гравитация\n'
    'Определение: Сила взаимного притяжения тел, которая удерживает планеты '
    'на орбите вокруг звёзд и заставляет предметы падать на Землю.\n\n'

    # Наш запрос — в том же формате, без ответа; модель продолжит
    'Понятие: нейронная сеть\n'
    'Определение:'
)

print('ПРОМПТ (few-shot + role):')
print(few_shot_prompt)
print()

result_few_shot = generate_text(
    few_shot_prompt,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.3,        # чуть ниже температура для более связного текста
    repetition_penalty=1.2,
)

# Вырезаем только сгенерированную часть (после «Определение:»)
generated_part = result_few_shot[len(few_shot_prompt):]

print('ТОЛЬКО СГЕНЕРИРОВАННОЕ ОПРЕДЕЛЕНИЕ:')
print('Определение:' + generated_part)

ПРОМПТ (few-shot + role):
Ты — опытный педагог, который объясняет сложные технические понятия простым и понятным языком.

Понятие: фотосинтез
Определение: Процесс, при котором растения используют солнечный свет для преобразования углекислого газа и воды в питательные вещества и кислород.

Понятие: гравитация
Определение: Сила взаимного притяжения тел, которая удерживает планеты на орбите вокруг звёзд и заставляет предметы падать на Землю.

Понятие: нейронная сеть
Определение:

ТОЛЬКО СГЕНЕРИРОВАННОЕ ОПРЕДЕЛЕНИЕ:
Определение: Биологическая структура, обеспечивающая передачу информации между neurons через электрическую связь.

Сложное слово: барьер

Объяснить каждое понятие по-другому.
Для каждого из этих понятий выделить 3–4 примеров использования (примеры должны быть разными).

Пр


Модель корректно воспроизвела формат: ответ начинается ровно с «Определение:», как и в примерах. Само определение содержательно верно — нейронная сеть действительно вдохновлена биологическими нейронами, передающими сигналы между собой.

Однако модель не остановилась на одном ответе: она продолжила генерировать новые задания («Сложное слово: барьер», «Объяснить каждое понятие по-другому...»), воспроизводя паттерн учебного текста. Это типичное поведение base-модели — она не различает «я ответила» и «текст продолжается», а просто предсказывает следующий наиболее вероятный токен.

### Сравнение подходов

| | Zero-shot | Few-shot + Role |
|---|---|---|
| **Длина промпта** | Короткий | Длиннее (роль + примеры) |
| **Контроль формата** | Нет | Да — модель воспроизводит формат примеров |
| **Стиль ответа** | Непредсказуемый | Управляемый через роль |
| **Риск «ухода в сторону»** | Высокий | Низкий |
| **Когда использовать** | Простые продолжения текста | Когда нужен конкретный формат |

**Ключевой вывод:** для base-моделей (не fine-tuned на инструкции) few-shot — это основной способ получить нужный формат ответа. Модель «видит» паттерн «Вопрос → Ответ» в примерах и воспроизводит его для нового запроса. Это работает потому, что модель обучена предсказывать следующий токен — а наиболее вероятным продолжением шаблона является ответ в том же формате.